# Proyecto de Creación de una Base de Datos Relacional

## 1. Objetivo del proyecto

Construir una base de datos relacional, normalizada, funcional y escalable en PostgreSQL a partir de cinco archivos CSV que contienen información sobre los alumnos y profesores de una escuela de bootcamps.

El objetivo no es simplemente almacenar los datos, sino organizarlos correctamente para:

- Evitar redundancias.
- Garantizar la integridad de los datos.
- Facilitar las consultas SQL.
- Permitir el crecimiento futuro de la base de datos.
- Facilitar el acceso desde aplicaciones externas.
- Automatizar el proceso de transformación e ingesta de datos.

---

## 2. Situación de partida

Disponemos de cinco archivos CSV con información sin normalizar.

| Archivo | Contenido | Registros |
|---|---|---:|
| clase_1.csv | Alumnos de Data Science | 15 |
| clase_2.csv | Alumnos de Data Science | 10 |
| clase_3.csv | Alumnos de Full Stack | 14 |
| clase_4.csv | Alumnos de Full Stack | 13 |
| claustro.csv | Profesorado | 10 |
| **Total** | | **62** |

### Problemas identificados

- Información repetida, como campus y promociones.
- Proyectos almacenados en columnas independientes.
- Diferentes estructuras según la vertical.
- Ausencia de claves primarias y foráneas.
- Dificultad para incorporar nuevos proyectos sin modificar la estructura.
- Falta de relaciones explícitas entre alumnos y profesores.

---

## 3. Visión global del proceso

El proyecto seguirá un proceso ETL (Extract, Transform, Load), combinado con el diseño de una base de datos relacional.

```text
CSV originales
      |
      v
Carga de datos con pandas
      |
      v
Exploración, limpieza y unificación
      |
      v
Transformación de datos
      |
      v
Modelo Entidad-Relación
      |
      v
Modelo lógico
      |
      v
DataFrames normalizados
      |
      v
Creación de tablas en PostgreSQL
      |
      v
Ingesta mediante to_sql()
      |
      v
Validación mediante consultas SQL
      |
      v
Despliegue en servidor remoto
      |
      v
Documentación y presentación
```

---

## 4. Fases del proyecto

### Fase 1. Carga de datos

**¿Qué hacemos?**

Importamos los cinco archivos CSV mediante `pd.read_csv()` y creamos un DataFrame por archivo.

**¿Por qué?**

Pandas nos permite explorar y manipular los datos antes de introducirlos en PostgreSQL.

**Herramientas:**

- Python.
- Pandas.
- Jupyter Notebook.

**Resultado esperado:** cinco DataFrames con los datos originales.

---

### Fase 2. Exploración, limpieza y unificación

**¿Qué hacemos?**

- Analizamos las dimensiones de los DataFrames.
- Comprobamos los nombres de las columnas.
- Revisamos los tipos de datos.
- Detectamos valores nulos y duplicados.
- Identificamos la vertical de cada clase.
- Añadimos un identificador de clase.
- Unificamos los cuatro DataFrames de alumnos.

**¿Por qué?**

Necesitamos conocer la estructura y calidad de los datos antes de diseñar la base de datos.

También queremos trabajar con todos los alumnos de manera conjunta, conservando su procedencia.

**Funciones utilizadas:**

- `shape`
- `info()`
- `isnull()`
- `duplicated()`
- `unique()`
- `pd.concat()`

**Resultado esperado:** un DataFrame unificado de alumnos y otro de profesores.

---

### Fase 3. Transformación y preparación de datos

**¿Qué hacemos?**

Transformamos las columnas de proyectos en filas mediante la función `melt()`.

También estandarizamos los nombres de los proyectos y eliminamos las combinaciones de proyectos que no corresponden a la vertical del alumno.

**¿Por qué?**

Almacenar cada proyecto en una columna limita la escalabilidad de la base de datos.

#### Antes de la transformación

| Alumno | EDA | BBDD | ML |
|---|---|---|---|
| Juan | Apto | No Apto | Apto |
| Ana | Apto | Apto | Apto |

#### Después de la transformación

| Alumno | Proyecto | Resultado |
|---|---|---|
| Juan | EDA | Apto |
| Juan | BBDD | No Apto |
| Juan | ML | Apto |
| Ana | EDA | Apto |
| Ana | BBDD | Apto |
| Ana | ML | Apto |

Con esta nueva estructura, si en el futuro se incorpora un proyecto, solamente tendremos que añadir registros, sin modificar las columnas de la tabla.

**Funciones utilizadas:**

- `melt()`
- `dropna()`
- `str.replace()`
- `reset_index()`
- `value_counts()`

**Resultado esperado:** 260 evaluaciones correspondientes a 52 alumnos y cinco proyectos por alumno, siempre que no falten evaluaciones.

---

### Fase 4. Diseño del modelo Entidad-Relación (E/R)

**¿Qué hacemos?**

Identificamos las entidades existentes y definimos sus atributos, relaciones y cardinalidades.

#### Posibles entidades

- Alumnos.
- Profesores.
- Campus.
- Verticales.
- Promociones.
- Clases.
- Modalidades.
- Aulas.
- Roles.
- Proyectos.
- Matrículas.
- Evaluaciones.
- Asignaciones de profesores.

#### Ejemplos de relaciones

- Un campus puede tener varias clases.
- Una vertical puede tener varios proyectos.
- Una clase puede tener varios alumnos matriculados.
- Un alumno puede realizar varios proyectos.
- Un proyecto puede ser realizado por varios alumnos.
- Un profesor puede estar asociado a diferentes grupos.

**¿Por qué?**

Necesitamos representar cómo funciona la escuela antes de construir técnicamente la base de datos.

**Herramienta propuesta:** Draw.io.

**Resultado esperado:** diagrama Entidad-Relación con las entidades, atributos, relaciones y cardinalidades.

Las entidades propuestas deberán validarse y consensuarse entre todos los integrantes del equipo.

---

### Fase 5. Diseño del modelo lógico

**¿Qué hacemos?**

Convertimos el modelo Entidad-Relación en una estructura concreta de tablas.

Definimos:

- Nombre de cada tabla.
- Columnas.
- Tipos de datos.
- Claves primarias (PK).
- Claves foráneas (FK).
- Restricciones.

#### Ejemplo de tabla campus

| Campo | Tipo de dato | Restricción |
|---|---|---|
| id_campus | INTEGER | PRIMARY KEY |
| nombre | VARCHAR | NOT NULL, UNIQUE |

#### Ejemplo de tabla clase

| Campo | Tipo de dato | Restricción |
|---|---|---|
| id_clase | INTEGER | PRIMARY KEY |
| id_campus | INTEGER | FOREIGN KEY |
| fecha_comienzo | DATE | NOT NULL |

**¿Por qué?**

Necesitamos determinar cómo se almacenarán y relacionarán técnicamente los datos.

Una clave primaria identifica de manera única cada registro, mientras que una clave foránea permite establecer relaciones entre tablas.

**Herramientas propuestas:** dbdiagram.io o Draw.io.

**Resultado esperado:** diagrama del modelo lógico de la base de datos.

---

### Fase 6. Normalización y creación de DataFrames

**¿Qué hacemos?**

Separamos la información en diferentes DataFrames siguiendo el modelo lógico y asignamos identificadores para relacionarlos.

#### Ejemplo de datos sin normalizar

| Alumno | Campus |
|---|---|
| Juan | Madrid |
| Ana | Madrid |
| Pedro | Valencia |

#### DataFrame de campus normalizado

| id_campus | nombre |
|---|---|
| 1 | Madrid |
| 2 | Valencia |

#### DataFrame relacionado con campus

| Alumno | id_campus |
|---|---|
| Juan | 1 |
| Ana | 1 |
| Pedro | 2 |

Este ejemplo es una simplificación para entender la normalización. En el modelo definitivo, el campus podría relacionarse con el alumno mediante su matrícula y clase.

**¿Por qué?**

Queremos evitar almacenar la misma información repetidamente y mantener la coherencia de los datos.

Si cambia el nombre de un campus, solamente tendremos que modificar un registro.

**Funciones utilizadas:**

- `drop_duplicates()`
- `merge()`
- `rename()`
- `reset_index()`
- `to_datetime()`

**Resultado esperado:** un DataFrame por cada tabla definitiva de la base de datos.

---

### Fase 7. Creación de tablas en PostgreSQL

**¿Qué hacemos?**

Creamos las tablas definitivas mediante scripts SQL, respetando el modelo lógico.

Utilizaremos instrucciones y restricciones como:

- `CREATE TABLE`
- `PRIMARY KEY`
- `FOREIGN KEY`
- `NOT NULL`
- `UNIQUE`
- `CHECK`

**¿Por qué?**

Necesitamos garantizar que los datos cumplen las reglas establecidas en nuestro diseño.

Por ejemplo:

- Impedir que existan identificadores duplicados.
- Impedir que una evaluación pertenezca a un alumno inexistente.
- Evitar resultados de evaluación no permitidos.

**Herramientas:**

- PostgreSQL.
- pgAdmin.
- SQL.

**Resultado esperado:** base de datos relacional estructurada y preparada para recibir los datos.

---

### Fase 8. Conexión e ingesta de datos

**¿Qué hacemos?**

Conectamos Python con PostgreSQL mediante SQLAlchemy e importamos los DataFrames normalizados utilizando `to_sql()`.

#### Ejemplo de ingesta

```python
df_alumnos.to_sql(
    name="alumno",
    con=engine,
    if_exists="append",
    index=False
)
```

**¿Por qué?**

Automatizamos la carga de datos sin tener que escribir manualmente cientos de instrucciones `INSERT INTO`.

Importaremos primero las tablas independientes y después aquellas que contienen claves foráneas.

Utilizaremos `if_exists="append"` para conservar las tablas y restricciones creadas previamente mediante SQL.

**Herramientas:**

- Python.
- Pandas.
- SQLAlchemy.
- PostgreSQL.

**Resultado esperado:** base de datos con todos los registros importados y correctamente relacionados.

---

### Fase 9. Validación y consultas SQL

**¿Qué hacemos?**

Comprobamos que los datos se han importado correctamente, que las relaciones funcionan y que no se ha perdido información durante las transformaciones.

#### Posibles consultas de demostración

1. Número de alumnos por campus.
2. Número de alumnos por vertical.
3. Número de alumnos por promoción.
4. Porcentaje de aprobados por proyecto.
5. Proyecto con mayor porcentaje de suspensos.
6. Alumnos que han aprobado todos los proyectos.
7. Distribución del profesorado por campus.
8. Distribución del profesorado por vertical y rol.
9. Comparación de resultados entre promociones.

**¿Por qué?**

Una base de datos no está terminada simplemente porque contenga datos. Debemos demostrar que funciona correctamente y que permite obtener información útil.

**Herramientas:**

- SQL.
- PostgreSQL.
- pgAdmin.

**Resultado esperado:** script de consultas SQL para validar y demostrar el funcionamiento de la base de datos.

---

### Fase 10. Despliegue en un servidor remoto

**¿Qué hacemos?**

Alojamos la base de datos PostgreSQL en un servidor accesible desde Internet mediante conexiones autorizadas.

**¿Por qué?**

Hasta ahora trabajamos en localhost. Necesitamos que otras aplicaciones puedan conectarse a la base de datos sin depender de nuestro ordenador.

Esto permitirá que los compañeros de Full Stack puedan desarrollar aplicaciones que consuman los datos.

**Herramienta propuesta:** Render u otro proveedor compatible con PostgreSQL.

**Consideraciones de seguridad:**

- Proteger las credenciales de acceso.
- Utilizar variables de entorno.
- No publicar contraseñas en GitHub.
- No exponer datos personales reales sin autorización.
- Configurar los permisos de acceso.

**Resultado esperado:** base de datos remota funcional y accesible desde aplicaciones externas autorizadas.

---

### Fase 11. Documentación y presentación

**¿Qué hacemos?**

Organizamos el código, los diagramas y la documentación en un repositorio compartido de GitHub.

#### Estructura propuesta del repositorio

```text
Proyecto/
|
|-- data/
|-- notebooks/
|-- diagrams/
|-- sql/
|   |-- 01_create_tables.sql
|   |-- 02_queries.sql
|
|-- src/
|-- presentation/
|-- README.md
|-- requirements.txt
|-- .gitignore
|-- .env.example
```

Los archivos con datos personales reales y las credenciales deberán mantenerse fuera del repositorio público.

**¿Por qué?**

Queremos que el proyecto sea reproducible, comprensible y mantenible por cualquier integrante del equipo.

#### Contenido de la presentación técnica

- Objetivo del proyecto.
- Análisis de los datos originales.
- Transformaciones realizadas.
- Modelo Entidad-Relación.
- Modelo lógico.
- Normalización.
- Creación e ingesta de la base de datos.
- Demostración mediante consultas SQL.
- Conclusiones y posibilidades de ampliación.

**Resultado esperado:** proyecto completo, documentado y listo para entregar.

---

## 5. Seguimiento del proyecto

| Fase | Herramienta principal | Estado |
|---|---|---|
| 1. Carga de datos | Pandas | Completado |
| 2. Exploración y unificación | Pandas | Completado |
| 3. Transformación de datos | Pandas | En curso |
| 4. Modelo Entidad-Relación | Draw.io | Pendiente |
| 5. Modelo lógico | dbdiagram.io | Pendiente |
| 6. Normalización de DataFrames | Pandas | Pendiente |
| 7. Creación de tablas | PostgreSQL | Pendiente |
| 8. Conexión e ingesta | SQLAlchemy y to_sql() | Pendiente |
| 9. Validación y consultas | SQL y pgAdmin | Pendiente |
| 10. Despliegue remoto | Servidor PostgreSQL | Pendiente |
| 11. Documentación y presentación | GitHub | Pendiente |

---

## 6. Principios fundamentales del proyecto

| Principio | Objetivo |
|---|---|
| Normalización | Evitar redundancias e inconsistencias. |
| Integridad | Garantizar que las relaciones entre datos sean válidas. |
| Escalabilidad | Incorporar nuevos campus, aulas, proyectos, verticales y promociones. |
| Automatización | Transformar e importar datos mediante código reutilizable. |
| Reproducibilidad | Permitir que cualquier integrante pueda ejecutar el proyecto. |
| Accesibilidad | Permitir conexiones desde aplicaciones externas. |
| Seguridad | Proteger las credenciales y los datos personales. |
| Mantenibilidad | Facilitar futuras modificaciones y ampliaciones. |

---

## 7. Decisiones pendientes del equipo

Antes de cerrar el modelo relacional, debemos resolver las siguientes cuestiones:

- Cómo relacionar al profesorado con las clases.
- Cómo gestionar la modalidad de los alumnos, que no aparece en sus CSV.
- Cómo identificar correctamente cada promoción.
- Cómo incorporar las aulas sin inventar datos.
- Qué entidades y relaciones necesitamos realmente.
- Cómo garantizar la integridad de las evaluaciones.
- Qué servidor utilizaremos para el despliegue.

No debemos inventar información que no esté respaldada por los datos originales.

---

## 8. Entregables

- [ ] Diagrama del modelo Entidad-Relación.
- [ ] Diagrama del modelo lógico.
- [ ] Notebook de exploración y transformación.
- [ ] DataFrames normalizados.
- [ ] Scripts SQL de creación de tablas.
- [ ] Código de conexión e ingesta de datos.
- [ ] Base de datos funcional.
- [ ] Base de datos desplegada en un servidor.
- [ ] Consultas SQL de demostración.
- [ ] Repositorio de GitHub documentado.
- [ ] Presentación técnica.

**Fecha de entrega:** viernes 27 de noviembre de 2026 a las 09:00.

---

## 9. Conclusión

El proyecto consiste en desarrollar un proceso ETL (Extract, Transform, Load) combinado con el diseño y construcción de una base de datos relacional.

- **Extract:** extraemos los datos de los archivos CSV.
- **Transform:** exploramos, limpiamos, transformamos y normalizamos los datos mediante pandas.
- **Load:** cargamos los datos en PostgreSQL mediante SQLAlchemy y to_sql().

Finalmente, validamos la base de datos mediante consultas SQL y la desplegamos en un servidor para permitir su utilización desde aplicaciones externas.

La finalidad es construir una solución que no solo responda a las necesidades actuales de la escuela, sino que también permita su crecimiento futuro.
